# CS229 L08 — Neural Networks: Backpropagation

**Video:** Spring 2026 · [YouTube](https://www.youtube.com/watch?v=ne2ngVAoMG8)  
**Instructor:** Tengyu Ma  
**Topics:** Differentiable circuits · The O(N) gradient theorem · Chain rule as Jacobian transpose · Two-phase backprop · Linear layer backward · Activation backward · Hessian-vector products

---

> 📌 *Lecture:* "This is kind of like the most technical question you can ask in deep learning these days — how do you do backpropagation. It's a non-trivial algorithm that is needed for every training job under the hood, even though you don't have to do it yourself these days. It's called backpropagation or auto-differentiation. But I think it's good to understand because some of the decisions you make depend fundamentally on how it works."

---

## Roadmap

| Section | Key Idea |
|---|---|
| 1. The problem | How to compute ∇_θ J efficiently for a deep network |
| 2. Differentiable circuits | Computational graphs as reusable arithmetic |
| 3. The O(N) theorem | Gradient costs the same as the forward pass |
| 4. Chain rule: Jacobian view | dJ/dZ = J_G^T · dJ/dU — only depends on G, not F |
| 5. Two-phase backprop | Phase 1: activation gradients · Phase 2: parameter gradients |
| 6. Backward: linear layer | dJ/dZ = W^T dJ/dU · dJ/dW = dJ/dU · Z^T |
| 7. Backward: activations | Diagonal Jacobian → elementwise multiply |
| 8. Full MLP backprop | End-to-end from scratch, verified against PyTorch |
| 9. Hessian-vector products | Advanced: second-order methods via double backprop |


## 1. The Problem

From L07, we have a multi-layer network $h_\theta(x)$ and a loss:

$$
J(\theta) = \frac{1}{n} \sum_{i=1}^n J_i(\theta)
$$

SGD requires computing $\nabla_\theta J_i(\theta)$ — the gradient of the loss for one example with respect to **all** parameters $\theta = \{W^{[1]}, b^{[1]}, W^{[2]}, b^{[2]}, \ldots\}$.

**Naïve approach:** finite differences. For each parameter $\theta_k$:

$$
\frac{\partial J}{\partial \theta_k} \approx \frac{J(\theta + \varepsilon e_k) - J(\theta)}{\varepsilon}
$$

**Cost:** $O(P)$ forward passes for $P$ parameters. A GPT-4 scale model has $P \sim 10^{12}$ — completely infeasible.

**Backpropagation:** computes all $P$ gradients in **one** backward pass — same cost as one forward pass.

> 📌 *Lecture:* "Both the forward pass (evaluating the loss) and the gradient can be done in big-O of the number of parameters. That's the fundamental complexity of computing these gradients."


## 2. Differentiable Circuits

### Computational graph

Any function built from **arithmetic operations** (+, −, ×, ÷) and **elementary functions** (exp, log, sin, sigmoid, ...) can be represented as a **differentiable circuit** — a directed acyclic graph (DAG) where:

- **Nodes** = intermediate values
- **Edges** = operations
- **Leaves** = inputs/parameters
- **Root** = scalar output (the loss $J$)

**Key advantage over formulas:** intermediate values can be **reused**. If $x_1 + x_2$ appears twice in the computation, you compute it once and reuse the node — you don't expand it twice.

### Example: $J = (x_1 + x_2) \cdot x_3 + (x_1 + x_2) \cdot x_4$

```
x1, x2  →  [+]  →  u1  →  [× x3]  →  v1  →  [+]  →  J
                    u1  →  [× x4]  →  v2  ↗
```

$u_1 = x_1 + x_2$ is computed **once** and fed to two downstream nodes.

**Size** of the circuit = number of operations. Neural networks are differentiable circuits where size ≈ number of parameters (matrix multiplications dominate).


## 3. The O(N) Gradient Theorem

### Theorem (informal)

Let $f: \mathbb{R}^L \to \mathbb{R}$ be a real-valued function computable by a differentiable circuit of size $N$. Then:

$$
\nabla_x f(x) \in \mathbb{R}^L \quad \text{is also computable in } O(N) \text{ time}
$$

**In words:** the gradient of $f$ can be computed in the same time as $f$ itself, by a circuit of the same size.

### Applied to neural networks

| Quantity | Corresponds to | Cost |
|---|---|---|
| Function $f$ | Loss $J_i(\theta)$ | O(P) — forward pass |
| Input $x$ | Parameters $\theta$ | — |
| Input dimension $L$ | Number of parameters $P$ | — |
| $\nabla_x f$ | $\nabla_\theta J_i$ | O(P) — backward pass |

**Why size ∝ parameters in neural networks?** For a matrix multiply $W \in \mathbb{R}^{m \times d}$: parameters = $md$, operations to compute $Wz$ = $md$. Always equal.

### Forward vs. backward terminology

- **Forward pass:** compute $J_i(\theta)$ — evaluate the loss
- **Backward pass:** compute $\nabla_\theta J_i$ — evaluate the gradient
- **Both cost O(P).** This is what makes training tractable.

---

> 🎯 **Interview:** Why is backpropagation efficient — why does it not cost $O(P^2)$?
>
> **A:** Backprop exploits the chain rule to reuse intermediate computations. Each node in the computational graph only needs to compute a local Jacobian-transpose product — it doesn't need to know what came before or after it in the graph. This local computation costs the same as the forward operation at that node. Composing these local computations across all nodes gives the full gradient in O(P) total — same as one forward pass. The key insight is that the Markovian structure of the chain rule lets each module forget the rest of the network.

## 4. Chain Rule: The Jacobian View

### Setup

Consider a module $G: \mathbb{R}^M \to \mathbb{R}^N$ embedded in a larger computation:

$$
z \xrightarrow{G} u \xrightarrow{F} J
$$

where $z \in \mathbb{R}^M$, $u = G(z) \in \mathbb{R}^N$, $J \in \mathbb{R}$ (scalar loss).

**Given:** $\frac{\partial J}{\partial u} \in \mathbb{R}^N$ (gradient w.r.t. output of $G$, computed by layers above)  
**Want:** $\frac{\partial J}{\partial z} \in \mathbb{R}^M$ (gradient w.r.t. input of $G$)

### Chain rule formula

By the chain rule:

$$
\frac{\partial J}{\partial z_i} = \sum_{j=1}^N \frac{\partial J}{\partial u_j} \cdot \frac{\partial u_j}{\partial z_i} = \sum_{j=1}^N \frac{\partial J}{\partial u_j} \cdot \left(\frac{\partial G}{\partial z}\right)_{ji}
$$

Collecting all $i$:

$$
\boxed{\frac{\partial J}{\partial z} = \left(\frac{\partial G}{\partial z}\right)^T \frac{\partial J}{\partial u}}
$$

where $\frac{\partial G}{\partial z} \in \mathbb{R}^{N \times M}$ is the **Jacobian** of $G$.

### The Markovian property

> 📌 *Lecture:* "The magic is that this computation only needs information about $G$ and $z$ — but not $F$. So even if $F$ is a very very complex function, as long as I know $dJ/du$ I can forget about how I got all of that. What happens above doesn't matter at all. It's Markovian in some sense."

This is why each PyTorch module only needs to implement a **local** `backward()` function — it receives `dJ/du` from above, computes `dJ/dz` using only its own Jacobian, and passes it down. No global knowledge needed.

### Backward function signature

Every module $G$ provides:
- `forward(z)` → $u = G(z)$
- `backward(dJ_du, z)` → $\frac{\partial J}{\partial z} = J_G(z)^T \cdot \frac{\partial J}{\partial u}$

---

> 🎯 **Interview:** What does the backward function of a module receive and return?
>
> **A:** It receives `dJ/du` — the gradient of the loss w.r.t. the module's **output** — and returns `dJ/dz` — the gradient w.r.t. the module's **input**. It computes this via `dJ/dz = Jacobian(G, z)^T @ dJ/du`. The key point: it only needs the local Jacobian of its own operation and the cached value of `z` from the forward pass. It does not need to know anything about what comes before or after it in the network.

## 5. Two-Phase Backprop

A neural network layer computes $u = G(z; \theta)$ — it has both an **input** $z$ and **parameters** $\theta$. Backprop needs both $\frac{\partial J}{\partial z}$ (to continue the backward pass) and $\frac{\partial J}{\partial \theta}$ (to update the parameters).

### Full backward through a network

For a network: $x \xrightarrow{M_1(\theta_1)} u_1 \xrightarrow{M_2(\theta_2)} u_2 \xrightarrow{M_3(\theta_3)} \cdots \to J$

**Phase 1 — Activation gradients (sequential, right to left):**

$$
\frac{\partial J}{\partial u_K} \to \frac{\partial J}{\partial u_{K-1}} \to \cdots \to \frac{\partial J}{\partial u_1} \to \frac{\partial J}{\partial x}
$$

Each step uses the chain rule: $\frac{\partial J}{\partial u_{\ell-1}} = J_{M_\ell}^T \cdot \frac{\partial J}{\partial u_\ell}$

**Phase 2 — Parameter gradients (can be parallelized):**

Once $\frac{\partial J}{\partial u_\ell}$ is known for each layer $\ell$:

$$
\frac{\partial J}{\partial \theta_\ell} = \text{backward}_{M_\ell}\left(\frac{\partial J}{\partial u_\ell}, u_{\ell-1}\right)
$$

Each $\frac{\partial J}{\partial \theta_\ell}$ is **independent** once Phase 1 is complete — can be computed in parallel across layers.

### Memory management

> 📌 *Lecture:* "Once you have computed all of Phase 1, you can drop the calculations you've done there to free memory. The dependency between tensors is very subtle, but once you understand the dependency you can optimize which order to compute them and when to free the memory — and that depends on this computational graph."

This is exactly what PyTorch's memory allocator and `retain_graph` control.

---

> 🎯 **Interview:** Why does backprop need to cache intermediate activations from the forward pass?
>
> **A:** The backward function for each module needs both `dJ/du` (from above) and the input `z` that was used during the forward pass — because the Jacobian $\partial G/\partial z$ is evaluated at the specific `z` that was computed. For a linear layer, the backward for W requires the input activation. For an activation function, the backward requires the pre-activation value. PyTorch stores these in a computation graph during the forward pass so they're available when `.backward()` is called. Gradient checkpointing trades memory for compute by recomputing activations instead of caching them.

## 6. Backward: Linear Layer

### Forward

$$
G(z) = Wz + b, \quad W \in \mathbb{R}^{N \times M},\ z \in \mathbb{R}^M,\ b \in \mathbb{R}^N
$$

### Backward w.r.t. input $z$

The Jacobian of $G$ w.r.t. $z$: $\left(\frac{\partial G}{\partial z}\right)_{ji} = \frac{\partial (Wz+b)_j}{\partial z_i} = W_{ji}$

So $\frac{\partial G}{\partial z} = W \in \mathbb{R}^{N \times M}$, and:

$$
\boxed{\frac{\partial J}{\partial z} = W^T \frac{\partial J}{\partial u}}
$$

### Backward w.r.t. weights $W$

By chain rule, for entry $(i,j)$ of $W$:

$$
\frac{\partial J}{\partial W_{ij}} = \frac{\partial J}{\partial u_i} \cdot z_j
$$

Collecting all entries:

$$
\boxed{\frac{\partial J}{\partial W} = \frac{\partial J}{\partial u} \cdot z^T}
$$

This is an **outer product** — for a single example, $\frac{\partial J}{\partial W}$ is **rank-1**.

> 📌 *Lecture:* "This is called Hebb's rule in biology. The update to the synapse — the gradient — equals the gradient of the output side times the value on the input side. The gradient for one weight matrix is always rank-1 for one example."

### Backward w.r.t. bias $b$

$$
\boxed{\frac{\partial J}{\partial b} = \frac{\partial J}{\partial u}}
$$

### Summary

| What | Formula | Shape |
|---|---|---|
| dJ/dz | $W^T \cdot \text{dJ/du}$ | $(M,)$ |
| dJ/dW | $\text{dJ/du} \cdot z^T$ | $(N, M)$ — rank-1 per example |
| dJ/db | $\text{dJ/du}$ | $(N,)$ |

---

> 🎯 **Interview:** Derive the gradient of the loss w.r.t. the weight matrix W in a linear layer.
>
> **A:** For layer $u = Wz + b$, apply chain rule: $\partial J/\partial W_{ij} = \sum_k (\partial J/\partial u_k)(\partial u_k/\partial W_{ij})$. Since $(Wz)_k = \sum_l W_{kl}z_l$, we have $\partial (Wz)_k/\partial W_{ij} = z_j$ if $k=i$, else 0. So $\partial J/\partial W_{ij} = (\partial J/\partial u_i) \cdot z_j$. Stacking all $(i,j)$ pairs: $\partial J/\partial W = (\partial J/\partial u) \cdot z^T$ — an outer product. For a mini-batch of size $B$: $\partial J/\partial W = \frac{1}{B} \sum_{b=1}^B (\partial J/\partial u^{(b)}) \cdot (z^{(b)})^T$.

## 7. Backward: Activation Function

### Forward

$$
G(z) = \sigma(z) \quad \text{(elementwise)}, \quad u_i = \sigma(z_i)
$$

### Jacobian

Since $u_i$ only depends on $z_i$ (not $z_j$ for $j \neq i$), the Jacobian is **diagonal**:

$$
\left(\frac{\partial G}{\partial z}\right)_{ij} = \begin{cases} \sigma'(z_i) & i = j \\ 0 & i \neq j \end{cases}
$$

### Backward

Multiplying by a diagonal matrix = elementwise multiply:

$$
\boxed{\frac{\partial J}{\partial z} = \sigma'(z) \odot \frac{\partial J}{\partial u}}
$$

where $\odot$ is the elementwise (Hadamard) product and $\sigma'(z) = [\sigma'(z_1), \ldots, \sigma'(z_M)]$.

### Common activation derivatives

| Activation | $\sigma(t)$ | $\sigma'(t)$ |
|---|---|---|
| ReLU | $\max(t, 0)$ | $\mathbf{1}[t > 0]$ |
| Sigmoid | $\frac{1}{1+e^{-t}}$ | $\sigma(t)(1-\sigma(t))$ |
| Tanh | $\tanh(t)$ | $1 - \tanh^2(t)$ |
| GELU | $t \cdot \Phi(t)$ | $\Phi(t) + t\phi(t)$ |

**ReLU backward:** for $t > 0$, gradient passes through unchanged; for $t \leq 0$, gradient is zeroed. This is the "dead ReLU" problem — a neuron that's always negative never gets any gradient.

### Efficiency check

Forward activation: O(M) — apply $\sigma$ elementwise.  
Backward activation: O(M) — compute $\sigma'$ elementwise and multiply.  
Same cost. Theorem holds.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Manual implementation of all backward functions ---

class LinearLayer:
    def __init__(self, in_dim, out_dim):
        scale = np.sqrt(2.0 / in_dim)
        self.W = np.random.randn(out_dim, in_dim) * scale
        self.b = np.zeros(out_dim)
        self.cache = None

    def forward(self, z):
        self.cache = z                      # cache for backward
        return self.W @ z + self.b          # u = Wz + b

    def backward(self, dJ_du):
        z = self.cache
        dJ_dz = self.W.T @ dJ_du           # dJ/dz = W^T dJ/du
        self.dJ_dW = np.outer(dJ_du, z)    # dJ/dW = dJ/du · z^T  (rank-1)
        self.dJ_db = dJ_du                  # dJ/db = dJ/du
        return dJ_dz

    def update(self, lr):
        self.W -= lr * self.dJ_dW
        self.b -= lr * self.dJ_db


class ReLU:
    def __init__(self):
        self.cache = None

    def forward(self, z):
        self.cache = z
        return np.maximum(z, 0)

    def backward(self, dJ_du):
        z = self.cache
        return dJ_du * (z > 0)             # σ'(z) ⊙ dJ/du — diagonal Jacobian


class Sigmoid:
    def __init__(self):
        self.cache = None

    def forward(self, z):
        u = 1 / (1 + np.exp(-z))
        self.cache = u                     # cache σ(z) for backward
        return u

    def backward(self, dJ_du):
        u = self.cache
        return dJ_du * u * (1 - u)         # σ'(z) = σ(z)(1-σ(z))


class MSELoss:
    def forward(self, pred, target):
        self.diff = pred - target
        return np.mean(self.diff**2)

    def backward(self):
        return 2 * self.diff / self.diff.size


# --- Verify backward formulas against finite differences ---
np.random.seed(42)

def check_grad(layer, x, dJ_du, eps=1e-5):
    """Finite-difference check for dJ/dz."""
    # Analytical gradient
    _ = layer.forward(x)
    dJ_dz_analytical = layer.backward(dJ_du)

    # Numerical gradient
    dJ_dz_numerical = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy(); x_plus[i] += eps
        x_minus = x.copy(); x_minus[i] -= eps
        f_plus = np.dot(dJ_du, layer.forward(x_plus))
        f_minus = np.dot(dJ_du, layer.forward(x_minus))
        dJ_dz_numerical[i] = (f_plus - f_minus) / (2 * eps)

    err = np.max(np.abs(dJ_dz_analytical - dJ_dz_numerical))
    return err

x = np.random.randn(4)
dJ_du = np.random.randn(6)

linear = LinearLayer(4, 6)
relu = ReLU()
sigmoid = Sigmoid()

err_linear = check_grad(linear, x, dJ_du)
err_relu = check_grad(relu, x, np.random.randn(4))
err_sigmoid = check_grad(sigmoid, x, np.random.randn(4))

print("Gradient check (max abs error vs finite differences):")
print(f"  LinearLayer:  {err_linear:.2e}")
print(f"  ReLU:         {err_relu:.2e}")
print(f"  Sigmoid:      {err_sigmoid:.2e}")
print("\nAll should be < 1e-7 to pass.")

## 8. Full MLP: End-to-End Backprop from Scratch

Putting it together: a 2-layer MLP trained with manual backprop, verified against PyTorch autograd.

**Architecture:** $x \xrightarrow{L_1} \xrightarrow{\text{ReLU}} \xrightarrow{L_2} \hat{y}$  
**Loss:** MSE $(\hat{y} - y)^2$

**Forward:**
$$z_1 = W_1 x + b_1, \quad a_1 = \text{ReLU}(z_1), \quad \hat{y} = W_2 a_1 + b_2$$

**Backward (right to left):**
$$\frac{\partial J}{\partial \hat{y}} = 2(\hat{y} - y)/n$$
$$\frac{\partial J}{\partial a_1} = W_2^T \frac{\partial J}{\partial \hat{y}}, \quad \frac{\partial J}{\partial W_2} = \frac{\partial J}{\partial \hat{y}} \cdot a_1^T, \quad \frac{\partial J}{\partial b_2} = \frac{\partial J}{\partial \hat{y}}$$
$$\frac{\partial J}{\partial z_1} = \frac{\partial J}{\partial a_1} \odot \mathbf{1}[z_1 > 0]$$
$$\frac{\partial J}{\partial W_1} = \frac{\partial J}{\partial z_1} \cdot x^T, \quad \frac{\partial J}{\partial b_1} = \frac{\partial J}{\partial z_1}$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# --- Manual MLP ---
class MLP:
    def __init__(self, d, m, k):
        s1 = np.sqrt(2.0/d); s2 = np.sqrt(2.0/m)
        self.W1 = np.random.randn(m, d) * s1
        self.b1 = np.zeros(m)
        self.W2 = np.random.randn(k, m) * s2
        self.b2 = np.zeros(k)

    def forward(self, x):
        self.x  = x
        self.z1 = self.W1 @ x + self.b1
        self.a1 = np.maximum(self.z1, 0)         # ReLU
        self.yh = self.W2 @ self.a1 + self.b2
        return self.yh

    def backward(self, dJ_dyh):
        # Layer 2 backward
        self.dW2 = np.outer(dJ_dyh, self.a1)     # dJ/dW2 = dJ/dyh · a1^T
        self.db2 = dJ_dyh
        dJ_da1  = self.W2.T @ dJ_dyh             # dJ/da1 = W2^T dJ/dyh

        # ReLU backward
        dJ_dz1  = dJ_da1 * (self.z1 > 0)         # σ'(z1) ⊙ dJ/da1

        # Layer 1 backward
        self.dW1 = np.outer(dJ_dz1, self.x)      # dJ/dW1 = dJ/dz1 · x^T
        self.db1 = dJ_dz1

    def step(self, lr):
        self.W2 -= lr * self.dW2; self.b2 -= lr * self.db2
        self.W1 -= lr * self.dW1; self.b1 -= lr * self.db1


# --- Training: approximate sin(x) ---
n, d, m, k = 200, 1, 32, 1
X = np.random.uniform(-np.pi, np.pi, (n, d))
Y = np.sin(X).reshape(n, k)

mlp = MLP(d, m, k)
lr = 0.01; losses = []

for step in range(2000):
    # Mini-batch SGD
    idx = np.random.choice(n, 32)
    total_dW2 = np.zeros_like(mlp.W2)
    total_db2 = np.zeros_like(mlp.b2)
    total_dW1 = np.zeros_like(mlp.W1)
    total_db1 = np.zeros_like(mlp.b1)
    batch_loss = 0

    for i in idx:
        yh = mlp.forward(X[i])
        diff = yh - Y[i]
        batch_loss += (diff**2).mean()
        dJ_dyh = 2 * diff / len(idx)
        mlp.backward(dJ_dyh)
        total_dW2 += mlp.dW2; total_db2 += mlp.db2
        total_dW1 += mlp.dW1; total_db1 += mlp.db1

    mlp.dW2 = total_dW2; mlp.db2 = total_db2
    mlp.dW1 = total_dW1; mlp.db1 = total_db1
    mlp.step(lr)
    losses.append(batch_loss / 32)

# --- Plot ---
x_test = np.linspace(-np.pi, np.pi, 200)
y_pred = np.array([mlp.forward(np.array([xi])) for xi in x_test]).flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_test, np.sin(x_test), 'k--', lw=2, label='sin(x) target')
axes[0].plot(x_test, y_pred, 'C0', lw=2, label='MLP (manual backprop)')
axes[0].set_title('Manual Backprop: sin(x) approximation'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(losses, 'C1', lw=1.5, alpha=0.8)
axes[1].set_yscale('log'); axes[1].set_title('Training Loss (MSE)')
axes[1].set_xlabel('Step'); axes[1].grid(True, alpha=0.3)

plt.suptitle('2-Layer MLP trained with manual backprop (no PyTorch)', fontsize=12)
plt.tight_layout(); plt.show()
print(f"Final loss: {losses[-1]:.6f}")

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# --- Verify our manual gradients match PyTorch autograd ---
np.random.seed(7)
d, m, k = 4, 8, 1

W1 = np.random.randn(m, d)
b1 = np.random.randn(m)
W2 = np.random.randn(k, m)
b2 = np.random.randn(k)
x  = np.random.randn(d)
y  = np.random.randn(k)

# --- Manual forward + backward ---
z1 = W1 @ x + b1
a1 = np.maximum(z1, 0)
yh = W2 @ a1 + b2
loss_np = ((yh - y)**2).mean()

dJ_dyh = 2 * (yh - y) / k
dW2_np = np.outer(dJ_dyh, a1)
db2_np = dJ_dyh
da1    = W2.T @ dJ_dyh
dz1    = da1 * (z1 > 0)
dW1_np = np.outer(dz1, x)
db1_np = dz1

# --- PyTorch autograd ---
W1_t = torch.tensor(W1, dtype=torch.float64, requires_grad=True)
b1_t = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
W2_t = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
b2_t = torch.tensor(b2, dtype=torch.float64, requires_grad=True)
x_t  = torch.tensor(x,  dtype=torch.float64)
y_t  = torch.tensor(y,  dtype=torch.float64)

z1_t = W1_t @ x_t + b1_t
a1_t = torch.relu(z1_t)
yh_t = W2_t @ a1_t + b2_t
loss_t = ((yh_t - y_t)**2).mean()
loss_t.backward()

# --- Compare ---
print("Gradient verification: manual backprop vs PyTorch autograd")
print("-" * 55)
for name, manual, torch_grad in [
    ('dW1', dW1_np, W1_t.grad.numpy()),
    ('db1', db1_np, b1_t.grad.numpy()),
    ('dW2', dW2_np, W2_t.grad.numpy()),
    ('db2', db2_np, b2_t.grad.numpy()),
]:
    err = np.max(np.abs(manual - torch_grad))
    status = '✓ PASS' if err < 1e-10 else '✗ FAIL'
    print(f"  {name:4s}: max error = {err:.2e}  {status}")

## 9. Hessian-Vector Products (Advanced)

The O(N) theorem can be applied **twice** to compute second-order information cheaply.

### Setup

The Hessian $H = \nabla^2 J(\theta) \in \mathbb{R}^{P \times P}$ has $P^2$ entries — too large to store ($P \sim 10^9$ for modern models → $10^{18}$ entries).

But the **Hessian-vector product** $Hv$ for any vector $v$ can be computed in $O(P)$:

$$
Hv = \nabla_\theta \left[ \nabla_\theta J(\theta) \cdot v \right]
$$

**Why:** define $g(\theta) = \nabla_\theta J(\theta) \cdot v$ (inner product with fixed $v$). Then:
1. $g(\theta)$ is a scalar function of $\theta$ — computable in $O(P)$ (one backward pass + one dot product)
2. $\nabla_\theta g = Hv$ — gradient of a scalar function → $O(P)$ by the theorem

Total cost: **two backward passes** = $O(P)$. Full Hessian would cost $O(P^2)$.

### Applications

| Method | Uses HVP for |
|---|---|
| Conjugate gradient | Solving $H^{-1}g$ without forming $H$ |
| K-FAC | Kronecker-factored curvature approximation |
| Meta-learning (MAML) | Differentiating through an optimization algorithm |
| Test-time training | Gradient of loss after $k$ gradient steps |
| Influence functions | Tracing training data influence on predictions |

> 📌 *Lecture:* "You can tune your learning rate by backpropagating through the algorithm. The loss function for optimization is: 'if I run this algorithm for 10 steps, what's my result?' That's defined by an algorithm, and you can still optimize it — even though algorithms are involved in defining the loss function."


In [ ]:
import torch

# Hessian-vector product via double backprop
torch.manual_seed(0)

# Simple model: scalar loss on 4 parameters
theta = torch.randn(4, requires_grad=True, dtype=torch.float64)
v = torch.randn(4, dtype=torch.float64)  # direction vector

def loss_fn(theta):
    # J(theta) = sum(theta^4) — non-trivial Hessian
    return (theta**4).sum()

# Method 1: Hessian-vector product via double backprop
J = loss_fn(theta)
grad = torch.autograd.grad(J, theta, create_graph=True)[0]
gv = (grad * v).sum()                          # g(theta) = ∇J · v
Hv_autograd = torch.autograd.grad(gv, theta)[0]  # ∇g = Hv

# Method 2: Explicit Hessian (only feasible for small theta)
H = torch.zeros(4, 4, dtype=torch.float64)
for i in range(4):
    theta_i_grad = torch.autograd.grad(loss_fn(theta), theta,
                                        create_graph=True)[0][i]
    H[i] = torch.autograd.grad(theta_i_grad, theta,
                                 retain_graph=True)[0].detach()
Hv_explicit = H @ v

# J = sum(theta^4) → ∇²J = diag(12*theta^2)
H_analytical = torch.diag(12 * theta.detach()**2)
Hv_analytical = H_analytical @ v

print("Hessian-vector product comparison:")
print(f"  Analytical  Hv: {Hv_analytical.numpy().round(4)}")
print(f"  Explicit    Hv: {Hv_explicit.numpy().round(4)}")
print(f"  Double-bp   Hv: {Hv_autograd.detach().numpy().round(4)}")
err = (Hv_autograd.detach() - Hv_analytical).abs().max().item()
print(f"\nMax error (double-bp vs analytical): {err:.2e}")
print("Double backprop computes Hv in O(P) — no need to form the full H matrix.")

## Summary

### The backprop algorithm

```
FORWARD PASS:
  x → [L1: z1=W1x+b1] → [ReLU: a1=max(z1,0)] → [L2: yh=W2a1+b2] → J
  (cache z1, a1 for backward)

BACKWARD PASS (right to left):
  dJ/dyh = 2(yh-y)/n                    ← loss gradient
  dJ/dW2 = dJ/dyh · a1^T               ← outer product (rank-1)
  dJ/db2 = dJ/dyh
  dJ/da1 = W2^T · dJ/dyh               ← transpose multiply
  dJ/dz1 = dJ/da1 ⊙ 1[z1>0]           ← elementwise (diagonal Jacobian)
  dJ/dW1 = dJ/dz1 · x^T               ← outer product (rank-1)
  dJ/db1 = dJ/dz1
```

### Key formulas

| Module | Forward | Backward |
|---|---|---|
| Linear $u=Wz+b$ | $u = Wz + b$ | $\partial J/\partial z = W^T \partial J/\partial u$ |
| Linear (weights) | — | $\partial J/\partial W = \partial J/\partial u \cdot z^T$ |
| Activation $u=\sigma(z)$ | $u_i = \sigma(z_i)$ | $\partial J/\partial z = \sigma'(z) \odot \partial J/\partial u$ |

### One key insight

Each module's backward only needs:
1. `dJ/du` from the layer above
2. The cached input `z` from its own forward pass

No global information. This is why autograd frameworks work.

---

## External Resources

| Resource | What to read | Why |
|---|---|---|
| CS229 Notes Part 5 | §4 Backpropagation | Formal derivation with notation matching lectures |
| Goodfellow *Deep Learning* | Ch. 6.5 — Backpropagation | Rigorous treatment with computational graph formalism |
| Karpathy — micrograd | [github.com/karpathy/micrograd](https://github.com/karpathy/micrograd) | 100-line autograd engine — best hands-on resource |
| PyTorch docs | `torch.autograd` | How `create_graph`, `retain_graph`, HVP work |
| Pearlmutter 1994 | "Fast Exact Multiplication by the Hessian" | Original HVP paper |
| Baydin et al. 2018 | "Automatic Differentiation in Machine Learning: a Survey" | Full overview of forward vs reverse mode AD |
